In [23]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder


class DataPreprocessor:

    def __init__(self):

        self.users = None
        self.courses = None
        self.transactions = None
        self.data = None

        self.scaler = StandardScaler()
        self.encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

    ########################################################
    # LOAD DATA
    ########################################################

    def load_data(
        self,
        users_path,
        courses_path,
        transactions_path
    ):

        print("Loading datasets...")

        self.users = pd.read_csv(users_path)

        self.courses = pd.read_csv(courses_path)

        self.transactions = pd.read_csv(transactions_path)

        print("Users :", self.users.shape)
        print("Courses :", self.courses.shape)
        print("Transactions :", self.transactions.shape)

    ########################################################
    # REMOVE DUPLICATES
    ########################################################

    def remove_duplicates(self):

        self.users.drop_duplicates(inplace=True)

        self.courses.drop_duplicates(inplace=True)

        self.transactions.drop_duplicates(inplace=True)

    ########################################################
    # HANDLE MISSING VALUES
    ########################################################

    def handle_missing_values(self):

    # USERS
        for col in self.users.columns:
    
            if self.users[col].dtype == "object":
                self.users[col] = self.users[col].fillna(
                    self.users[col].mode()[0]
                )
    
            else:
                self.users[col] = self.users[col].fillna(
                    self.users[col].median()
                )

    # COURSES
        for col in self.courses.columns:
    
            if self.courses[col].dtype == "object":
                self.courses[col] = self.courses[col].fillna(
                    self.courses[col].mode()[0]
                )
    
            else:
                self.courses[col] = self.courses[col].fillna(
                    self.courses[col].median()
                )

    # TRANSACTIONS
        for col in self.transactions.columns:
    
            if self.transactions[col].dtype == "object":
                self.transactions[col] = self.transactions[col].fillna(
                    self.transactions[col].mode()[0]
                )
    
            else:
                self.transactions[col] = self.transactions[col].fillna(
                    self.transactions[col].median()
                )
  

    ########################################################
    # CLEAN TEXT
    ########################################################

    def clean_text(self):

        dfs = [self.users, self.courses]

        for df in dfs:

            object_cols = df.select_dtypes(include="object").columns

            for col in object_cols:

                df[col] = (
                    df[col]
                    .astype(str)
                    .str.strip()
                    .str.title()
                )

    ########################################################
    # DATE CONVERSION
    ########################################################

    def convert_date(self):

        self.transactions["TransactionDate"] = pd.to_datetime(

            self.transactions["TransactionDate"],
    
            dayfirst=True,
    
            errors="coerce"

    )
    ########################################################
    # REMOVE INVALID AMOUNT
    ########################################################

    def clean_amount(self):

        if "Amount" in self.transactions.columns:

            self.transactions = self.transactions[
                self.transactions["Amount"] >= 0
            ]

    ########################################################
    # MERGE DATA
    ########################################################

    def merge_data(self):

        print("Cleaning IDs...")
    
        self.transactions["CourseID"] = (
            self.transactions["CourseID"]
            .astype(str)
            .str.strip()
            .str.upper()
        )
    
        self.courses["CourseID"] = (
            self.courses["CourseID"]
            .astype(str)
            .str.strip()
            .str.upper()
        )
    
        self.transactions["UserID"] = (
            self.transactions["UserID"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        self.users["UserID"] = (
            self.users["UserID"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        print("\nTransaction CourseIDs")
    
        print(self.transactions["CourseID"].head())
    
        print("\nCourse CourseIDs")
    
        print(self.courses["CourseID"].head())
    
        temp = pd.merge(
    
            self.transactions,
    
            self.courses,
    
            on="CourseID",
    
            how="left"
    
        )

        self.data = pd.merge(
    
            temp,
    
            self.users,
    
            on="UserID",
    
            how="left"
    
        )
    
        print("\nMerged Shape :", self.data.shape)
    
        print("\nMissing Course Names")
    
        print(self.data["CourseName"].isna().sum())

        return self.data

    ########################################################
    # ENCODE
    ########################################################

    def encode(self, df, categorical_columns):

        encoded = self.encoder.fit_transform(
            df[categorical_columns]
        )

        encoded = pd.DataFrame(

            encoded,

            columns=self.encoder.get_feature_names_out(
                categorical_columns
            )

        )

        df = df.drop(columns=categorical_columns)

        df = pd.concat(

            [df.reset_index(drop=True),
             encoded.reset_index(drop=True)],

            axis=1

        )

        return df

    ########################################################
    # SCALE
    ########################################################

    def scale(self, df, numerical_columns):

        df[numerical_columns] = self.scaler.fit_transform(
            df[numerical_columns]
        )

        return df

    ########################################################
    # SAVE OBJECTS
    ########################################################

    def save_objects(self):

        if not os.path.exists("models"):

            os.mkdir("models")

        joblib.dump(

            self.scaler,

            "models/scaler.pkl"

        )

        joblib.dump(

            self.encoder,

            "models/encoder.pkl"

        )

        print("Scaler Saved")

        print("Encoder Saved")

    ########################################################
    # COMPLETE PREPROCESSING
    ########################################################

    def preprocess(

        self,

        users_path,

        courses_path,

        transactions_path

    ):

        self.load_data(

            users_path,

            courses_path,

            transactions_path

        )

        self.remove_duplicates()

        self.handle_missing_values()

        self.clean_text()

        self.convert_date()

        self.clean_amount()

        merged = self.merge_data()

        return merged


############################################################
# MAIN
############################################################

if __name__ == "__main__":

    processor = DataPreprocessor()

    df = processor.preprocess(

        users_path="dataset_EduPro/EduPro Online Platform.xlsx - Users.csv",

        courses_path="dataset_EduPro/EduPro Online Platform.xlsx - Courses.csv",

        transactions_path="dataset_EduPro/EduPro Online Platform.xlsx - Transactions.csv"

    )

    print(df.head())

    print(df.columns)

    # Save merged dataset
    df.to_csv("merged_data.csv", index=False)
    
    print("Merged dataset saved successfully!")

Loading datasets...
Users : (3000, 5)
Courses : (60, 8)
Transactions : (10000, 7)
Cleaning IDs...

Transaction CourseIDs
0    CR00016
1    CR00037
2    CR00019
3    CR00048
4    CR00060
Name: CourseID, dtype: object

Course CourseIDs
0    CR00001
1    CR00002
2    CR00003
3    CR00004
4    CR00005
Name: CourseID, dtype: object

Merged Shape : (10000, 18)

Missing Course Names
0
  TransactionID  UserID CourseID TransactionDate  Amount  PaymentMethod  \
0       TT00001  U00003  CR00016      2025-10-25     0.0         PayPal   
1       TT00002  U00003  CR00037      2025-01-13     0.0         PayPal   
2       TT00003  U00003  CR00019      2025-03-28     0.0  Bank Transfer   
3       TT00004  U00004  CR00048      2025-06-02     0.0  Bank Transfer   
4       TT00005  U00004  CR00060      2025-08-10     0.0         PayPal   

  TeacherID         CourseName           CourseCategory CourseType  \
0   TC00040  Digital Marketing                Marketing       Free   
1   TC00040   Scrum Essentia